# OMEGA GPT-1: Train on Google Colab GPU
**Project:** OMEGA GPT-1 Custom Decoder-Only Transformer Language Model  
**Objective:** Train an original GPT model from scratch using a free Google Colab GPU (T4 / A100 / V100).

---
### Instructions to enable GPU in Colab:
1. Go to **Runtime** in the top menu -> **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or A100 if subscribed to Colab Pro).
3. Click **Save**.
4. Run each cell sequentially below (`Shift + Enter`).

## 1. Verify GPU Hardware Acceleration

In [ ]:
import torch
print(f"PyTorch Version : {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available  : {gpu_available}")

if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU      : {gpu_name}")
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print("SUCCESS: GPU is active and ready for high-speed training!")
else:
    print("WARNING: Running on CPU. Please go to Runtime -> Change runtime type -> T4 GPU for 50x faster training.")

## 2. Install Required Packages

In [ ]:
!pip install -q torch numpy tokenizers matplotlib tqdm tensorboard

## 3. Project Workspace Setup
This cell checks if the project files are present. If not, it allows you to upload `omega-gpt.zip` directly.

In [ ]:
import os
import sys

# Check if main.py is already in current folder
if not os.path.exists("main.py"):
    if os.path.exists("omega-gpt"):
        %cd omega-gpt
    elif os.path.exists("omega-gpt.zip"):
        !unzip -q omega-gpt.zip
        if os.path.exists("omega-gpt"):
            %cd omega-gpt
    else:
        print("Please upload omega-gpt.zip using the file upload button below:")
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                !unzip -q "$fname"
                if os.path.exists("omega-gpt"):
                    %cd omega-gpt
                break

print(f"Current Working Directory: {os.getcwd()}")
!python main.py setup

## 4. Mount Google Drive (Backup Checkpoints)
Automatically saves model checkpoints to your Google Drive so you never lose training progress.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_DIR = '/content/drive/MyDrive/OmegaGPT_Colab_Checkpoints'
    os.makedirs(GDRIVE_DIR, exist_ok=True)
    print(f"Google Drive connected! Checkpoints will be backed up to: {GDRIVE_DIR}")
except Exception as e:
    print(f"Google Drive mount skipped: {e}")

## 5. Train Custom BPE Tokenizer From Scratch

In [ ]:
!python main.py train-tokenizer --vocab_size 1200

## 6. Preprocess, Tokenize & Split Dataset

In [ ]:
!python main.py prepare-data --train_ratio 0.9

## 7. Model Diagnostics & Parameter Inspection

In [ ]:
!python main.py info --profile small

## 8. Train Omega GPT-1 on Colab GPU
Trains with mixed precision (AMP FP16) and fused AdamW on CUDA. Runs ~100x faster than CPU.

In [ ]:
# Train Small model profile for 2,000 steps with GPU acceleration
!python main.py train --profile small --max_steps 2000 --batch_size 64 --device auto

# Copy best checkpoint to Google Drive backup if mounted
if os.path.exists('/content/drive/MyDrive/OmegaGPT_Colab_Checkpoints'):
    !cp outputs/checkpoints/omega_gpt1_best.pth /content/drive/MyDrive/OmegaGPT_Colab_Checkpoints/
    print("Backed up best checkpoint to Google Drive.")

## 9. View Training Loss and Perplexity Graphs

In [ ]:
from IPython.display import Image, display
graph_file = "outputs/graphs/training_metrics.png"
if os.path.exists(graph_file):
    display(Image(graph_file))
else:
    print("Graph file will appear after evaluation steps.")

## 10. Evaluate Validation Perplexity

In [ ]:
!python main.py evaluate

## 11. Generate Text on GPU

In [ ]:
# Test prompt generation
!python main.py generate --prompt "Artificial intelligence is" --tokens 150 --temperature 0.75

In [ ]:
!python main.py generate --prompt "Transformers use attention mechanisms to" --tokens 150 --temperature 0.7

## 12. Download Trained Checkpoint to Your Local Machine

In [ ]:
from google.colab import files
checkpoint_to_download = "outputs/checkpoints/omega_gpt1_best.pth"
if os.path.exists(checkpoint_to_download):
    print(f"Downloading {checkpoint_to_download}...")
    files.download(checkpoint_to_download)
else:
    print(f"File {checkpoint_to_download} not found.")